라이브러리 불러오기

In [245]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import re
from sklearn.linear_model import LinearRegression

RandomForest

In [246]:
df = pd.read_csv('../data/processed/ebay_laptops_processed_all.csv')
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.head()

(4475, 37)
['source_row', 'price_min_usd', 'price_max_usd', 'price_usd', 'is_price_range', 'brand', 'model', 'release_year', 'condition', 'processor', 'processor_brand', 'processor_family', 'processor_speed_ghz', 'gpu', 'gpu_vendor', 'ram_gb', 'ssd_gb', 'hdd_gb', 'storage_type', 'storage_capacity_gb', 'screen_size_inch', 'resolution_width', 'resolution_height', 'os', 'device_type', 'color', 'has_seller_note', 'has_touchscreen', 'has_backlit_keyboard', 'has_bluetooth', 'has_webcam', 'has_wifi', 'ram_multi_value', 'ssd_multi_value', 'hdd_multi_value', 'screen_multi_value', 'core_spec_count']
source_row                int64
price_min_usd           float64
price_max_usd           float64
price_usd               float64
is_price_range             bool
brand                       str
model                       str
release_year            float64
condition                   str
processor                   str
processor_brand             str
processor_family            str
processor_speed_ghz

,source_row,price_min_usd,price_max_usd,price_usd,is_price_range,brand,model,release_year,condition,processor,...,has_touchscreen,has_backlit_keyboard,has_bluetooth,has_webcam,has_wifi,ram_multi_value,ssd_multi_value,hdd_multi_value,screen_multi_value,core_spec_count
0,2,303.68,303.68,303.68,False,chuwi,corebook x,2021.0,new,quad core,...,0,1,0,1,0,False,False,False,False,7
1,3,399.99,634.99,517.49,True,dell,dell latitude 7490,NaN,refurbished,intel core i7 8th gen,...,0,1,1,1,1,False,False,False,False,5
2,4,175.00,175.00,175.00,False,dell,dell latitude e5470,2019.0,used,intel core i5-6300u,...,1,1,1,1,1,False,False,False,False,7
3,5,84.99,84.99,84.99,False,hp,hp chromebook 11 g6,NaN,refurbished,intel celeron n,...,0,0,1,1,1,False,False,False,False,6
4,6,101.22,101.22,101.22,False,dell,various models,2015.0,refurbished,intel core i5 6th gen,...,0,0,0,1,1,False,False,False,False,7


Train/Test 분리 + 전처리 확인

In [247]:
[c for c in df.columns if "price" in c.lower()]

['price_min_usd', 'price_max_usd', 'price_usd', 'is_price_range']

In [249]:
price_cols = [c for c in df.columns if "price" in c.lower()]
X =df.drop(columns=price_cols)
y = df["price_min_usd"]

print(X.shape, y.shape)
X.head()

(4475, 33) (4475,)


,source_row,brand,model,release_year,condition,processor,processor_brand,processor_family,processor_speed_ghz,gpu,...,has_touchscreen,has_backlit_keyboard,has_bluetooth,has_webcam,has_wifi,ram_multi_value,ssd_multi_value,hdd_multi_value,screen_multi_value,core_spec_count
0,2,chuwi,corebook x,2021.0,new,quad core,other,other,3.8,intel iris plus graphics 655,...,0,1,0,1,0,False,False,False,False,7
1,3,dell,dell latitude 7490,NaN,refurbished,intel core i7 8th gen,intel,core_i7,4.2,intel uhd graphics 620,...,0,1,1,1,1,False,False,False,False,5
2,4,dell,dell latitude e5470,2019.0,used,intel core i5-6300u,intel,core_i5,2.4,intel hd graphics,...,1,1,1,1,1,False,False,False,False,7
3,5,hp,hp chromebook 11 g6,NaN,refurbished,intel celeron n,intel,celeron,2.4,intel hd graphics 500,...,0,0,1,1,1,False,False,False,False,6
4,6,dell,various models,2015.0,refurbished,intel core i5 6th gen,intel,core_i5,1.4,integrated,...,0,0,0,1,1,False,False,False,False,7


In [250]:
# 범주형 데이터 원핫인코딩 필요
cat_cols = X.select_dtypes(include="object").columns.tolist()
print("범주형 컬럼:", cat_cols)
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(X.shape)
X.head()

범주형 컬럼: ['brand', 'model', 'condition', 'processor', 'processor_brand', 'processor_family', 'gpu', 'gpu_vendor', 'storage_type', 'os', 'device_type', 'color']
(4475, 1912)


C:\Users\EZ\AppData\Local\Temp\ipykernel_8528\329843962.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns.tolist()


,source_row,release_year,processor_speed_ghz,ram_gb,ssd_gb,hdd_gb,storage_capacity_gb,screen_size_inch,resolution_width,resolution_height,...,color_standard,color_steam blue,color_steel gray,color_storm gray,color_teal,color_tech black,color_unknown,color_warm gold,color_white,color_yellow
0,2,2021.0,3.8,8.0,1024.0,512.0,1024.0,14.0,2160.0,1440.0,...,False,False,False,False,False,False,False,False,False,False
1,3,NaN,4.2,NaN,NaN,2048.0,2048.0,14.0,1920.0,1080.0,...,False,False,False,False,False,False,False,False,False,False
2,4,2019.0,2.4,16.0,500.0,500.0,500.0,14.0,1920.0,1080.0,...,False,False,False,False,False,False,False,False,False,False
3,5,NaN,2.4,4.0,NaN,16.0,16.0,11.6,1366.0,768.0,...,False,False,False,False,False,False,False,False,False,False
4,6,2015.0,1.4,8.0,256.0,NaN,256.0,12.5,1366.0,768.0,...,False,False,False,False,False,False,True,False,False,False


In [251]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(3580, 1912) (895, 1912)


RandomForestRegressor 학습 + 평가 지표

In [257]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

preds = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("===== RandomForest =====")
print(f"RF_MAE: {mae}")
print(f"RF_MSE: {mse}")
print(f"RF_RMSE: {rmse}")
print(f"RF_R2: {r2}")

===== RandomForest =====
RF_MAE: 99.5085811173184
RF_MSE: 41967.97629712134
RF_RMSE: 204.86087058567662
RF_R2: 0.7565627621933908


----
XGBoost

In [258]:
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
)

# ========================================================
# XGBoost가 허용하지 않는 특수문자를 언더스코어로 치환
X_train.columns = [re.sub(r"[\[\]<>]", "_", col) for col in X_train.columns]
X_test.columns = [re.sub(r"[\[\]<>]", "_", col) for col in X_test.columns]
# =========================================================

xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_mse = mean_squared_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_r2 = r2_score(y_test, xgb_preds)

print("===== XGBoost =====")
print(f"XGB_MAE: {xgb_mae}")
print(f"XGB_MSE: {xgb_mse}")
print(f"XGB_RMSE: {xgb_rmse}")
print(f"XGB_R2: {xgb_r2}")

===== XGBoost =====
XGB_MAE: 103.48927270891413
XGB_MSE: 37484.738099866605
XGB_RMSE: 193.60975724344732
XGB_R2: 0.7825679980771052


---
LinearRegression

In [259]:
X_train.isnull().sum()[X_train.isnull().sum() > 0]

release_year           3126
processor_speed_ghz    1872
ram_gb                 2280
ssd_gb                 1801
hdd_gb                 2090
storage_capacity_gb    1661
screen_size_inch       1461
resolution_width       1989
resolution_height      1989
dtype: int64

In [260]:
# 학습 데이터 기준으로 결측치 채우기
X_train_filled = X_train.fillna(X_train.mean())
X_test_filled = X_test.fillna(X_train.mean())  # test도 train의 평균값 기준으로 채워야 함 (data leakage 방지)

In [261]:
lr_model = LinearRegression()
lr_model.fit(X_train_filled, y_train)

lr_preds = lr_model.predict(X_test_filled)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

print("===== LinearRegression =====")
print(f"LR_MAE: {lr_mae}")
print(f"LR_RMSE: {lr_rmse}")
print(f"LR_R2: {lr_r2}")

===== LinearRegression =====
LR_MAE: 151.46702063488098
LR_RMSE: 278.4840639603085
LR_R2: 0.5501479750600056
